# E035 — Feature-Level Fusion: MFCC + LPCC Concatenation

**Motivation:** E022 tried score-level fusion of MFCC+LPCC but collapsed to LPCC alone (w=0.07). Feature-level fusion (concatenating MFCC+LPCC features before UBM) may capture complementary information better.

**Hypothesis:** Concatenating MFCC and LPCC features (39d + 39d = 78d) before UBM training will improve over LPCC alone by capturing both spectral envelope (MFCC) and vocal tract resonances (LPCC).

**Approach:**
1. Extract MFCC 13+Δ+ΔΔ (39d) and LPCC 13+Δ+ΔΔ (39d)
2. Concatenate → 78d feature vector
3. Train UBM-32 on 78d features
4. MAP adapt target model
5. Compare to LPCC-only baseline (E025)

**Variants:**
- `lpcc_only`: E025 baseline (39d)
- `mfcc+lpcc`: Concatenated (78d)
- `mfcc+lpcc+pca`: PCA-reduced to 50d before UBM

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E025_REF = {'mean_eer': 1.94, 'std_eer': 1.57}

In [ ]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_mfcc(y, sr):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat = np.vstack([mfcc, delta, delta2]).T
    feat -= feat.mean(axis=0)
    return feat

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def _train_ubm(X):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type="diag",
                           max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

def train_and_score(train_df, val_df, data_dir, feature_type, seed):
    """Train UBM-MAP and score validation samples."""
    rng = np.random.default_rng(seed)
    
    X_list, y_list = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        
        if feature_type == 'lpcc':
            feat = _extract_lpcc(y_wav, sr)
            wavs = [y_wav, _aug_pitch(y_wav, sr, rng)]
            for y_aug in wavs:
                f = _extract_lpcc(y_aug, sr)
                X_list.append(f); y_list.extend([row["label"]] * len(f))
        
        elif feature_type == 'mfcc+lpcc':
            mfcc = _extract_mfcc(y_wav, sr)
            lpcc = _extract_lpcc(y_wav, sr)
            feat = np.hstack([mfcc, lpcc])  # 78d
            wavs = [y_wav, _aug_pitch(y_wav, sr, rng)]
            for y_aug in wavs:
                mfcc_aug = _extract_mfcc(y_aug, sr)
                lpcc_aug = _extract_lpcc(y_aug, sr)
                f = np.hstack([mfcc_aug, lpcc_aug])
                X_list.append(f); y_list.extend([row["label"]] * len(f))
        
        elif feature_type == 'mfcc+lpcc+pca':
            mfcc = _extract_mfcc(y_wav, sr)
            lpcc = _extract_lpcc(y_wav, sr)
            feat = np.hstack([mfcc, lpcc])
            wavs = [y_wav, _aug_pitch(y_wav, sr, rng)]
            for y_aug in wavs:
                mfcc_aug = _extract_mfcc(y_aug, sr)
                lpcc_aug = _extract_lpcc(y_aug, sr)
                f = np.hstack([mfcc_aug, lpcc_aug])
                X_list.append(f); y_list.extend([row["label"]] * len(f))
    
    X = np.vstack(X_list)
    y = np.array(y_list)
    
    # Apply PCA if requested
    if feature_type == 'mfcc+lpcc+pca':
        pca = PCA(n_components=50, random_state=SEED)
        X = pca.fit_transform(X)
    
    # Train UBM on non-target
    ubm = _train_ubm(X[y == 0])
    adapted = _map_adapt(ubm, X[y == 1])
    
    # Score validation
    scores = []
    for _, row in val_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        
        if feature_type == 'lpcc':
            feat = _extract_lpcc(y_wav, sr)
        else:
            mfcc = _extract_mfcc(y_wav, sr)
            lpcc = _extract_lpcc(y_wav, sr)
            feat = np.hstack([mfcc, lpcc])
            if feature_type == 'mfcc+lpcc+pca':
                feat = pca.transform(feat.reshape(1, -1))[0]
        
        score = adapted.score(feat) - ubm.score(feat)
        scores.append(score)
    
    return np.array(scores)

print('Feature functions defined')

## 2. Cross-validation

In [ ]:
feature_types = ['lpcc', 'mfcc+lpcc', 'mfcc+lpcc+pca']
results = {}

for feat_type in feature_types:
    print(f"\n=== {feat_type} ===")
    fold_eers = []
    oof_scores = np.full(len(manifest), np.nan)
    
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        
        scores = train_and_score(train_df, val_df, DATA, feat_type, seed_f)
        oof_scores[val_idx] = scores
        
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    
    results[feat_type] = {
        'fold_eers': fold_eers,
        'mean': np.mean(fold_eers),
        'std': np.std(fold_eers),
    }
    print(f"  Mean ± Std: {np.mean(fold_eers):.2f} ± {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for feat_type, res in results.items():
    print(f"{feat_type:15s}: {res['mean']:5.2f} ± {res['std']:5.2f}%")

## 3. Conclusion

Feature-level fusion: [TBD]
Decision: [Adopt/Reject]